# 🗑️ TrashNet Dataset Exploration & Data Hygiene

This notebook explores the **TrashNet** waste classification dataset:
- Verification of stratified split manifests
- Class balance and representation
- Visualizing sample waste images across all 6 classes
- Image dimensions, channels, and aspect ratio consistency

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path("..")
manifest_path = PROJECT_ROOT / "data" / "splits" / "split_manifest.csv"
mapping_path = PROJECT_ROOT / "data" / "splits" / "class_mapping.json"

df = pd.read_csv(manifest_path)
with open(mapping_path) as f:
    mapping = json.load(f)

print(f"Total Images: {len(df)}")
print(f"Classes ({mapping['num_classes']}): {mapping['classes']}")
df.head()

## 1. Class Distribution Analysis
Analyzing the distribution of images across the 6 categories and verifying stratification across train, validation, and test sets.

In [ ]:
split_table = pd.crosstab(df["class_name"], df["split"], margins=True)[["train", "val", "test", "All"]]
split_table["percent"] = (split_table["All"] / len(df) * 100).round(2)
print("Per-class distribution across splits:")
display(split_table)

plt.figure(figsize=(9, 4))
ax = sns.countplot(data=df, x="class_name", hue="split", palette="Set2")
plt.title("Class Distribution Stratified by Split")
plt.xlabel("Waste Category")
plt.ylabel("Image Count")
plt.tight_layout()
plt.show()

## 2. Visualize Sample Images from Each Category

In [ ]:
classes = mapping["classes"]
samples_per_class = 4

fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(12, 14))
for row_idx, cls in enumerate(classes):
    sample_paths = df[df["class_name"] == cls].sample(samples_per_class, random_state=42)["filepath"]
    for col_idx, rel_path in enumerate(sample_paths):
        img_path = PROJECT_ROOT / rel_path
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = axes[row_idx, col_idx]
        ax.imshow(img_rgb)
        ax.axis("off")
        if col_idx == 0:
            ax.set_title(cls.capitalize(), fontsize=12, fontweight="bold", loc="left")
plt.suptitle("Sample Images from TrashNet Categories", fontsize=15, y=0.92)
plt.show()

## 3. Image Dimensions & Resolution Check

In [ ]:
print("Resolution distribution across dataset:")
res_counts = df.groupby(["width", "height", "channels"]).size().reset_index(name="count")
display(res_counts)

df["aspect_ratio"] = (df["width"] / df["height"]).round(3)
print("Aspect Ratio Distribution:")
print(df["aspect_ratio"].value_counts())